# How to Run Ingest a PDF into the Weaviate database in the Cloud
This notebook goes over an example case of scraping a PDF, chunking it, uploading it to a Weaviate database, and querying the database. It is designed to be run on data-int.lsst.cloud, and you must have access to that platform in order to successfully run it. While this example case is for a PDF, this workflow is designed to be used for any data source, so long as said data source can be converted to a Langchain document object.

In [ ]:
%pip install weaviate-client
%pip install langchain==0.3.20
%pip install openai==1.65.4
%pip install langchain-openai==0.3.7
%pip install langchain-weaviate==0.0.4
%pip install langchain-community==0.3.19
%pip install pymupdf
%pip install python-dotenv

In [ ]:
import os
from pathlib import Path
import time

import weaviate
from dotenv import load_dotenv
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_core.documents.base import Document
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters.character import RecursiveCharacterTextSplitter
from langchain_weaviate.vectorstores import WeaviateVectorStore
from weaviate.classes.init import Auth
from weaviate.classes.query import MetadataQuery

## Connect to Weaviate
Test the connection to weaviate. Make sure to have a .env file in the same directory with OPENAI_API_KEY and WEAVIATE_API_KEY set to the correct values.

In [ ]:
load_dotenv()
openai_api_key = os.getenv("OPENAI_API_KEY")
weaviate_api_key = os.getenv("WEAVIATE_API_KEY")
http_host = "weaviate-headless.rubin-rag.svc.cluster.local"
grpc_host = "weaviate-grpc.rubin-rag.svc.cluster.local"

if openai_api_key is None:
    raise ValueError("OPENAI_API_KEY environment variable is not set")
if weaviate_api_key is None:
    raise ValueError("WEAVIATE_API_KEY environment variable is not set")
if http_host is None:
    raise ValueError("HTTP_HOST environment variable is not set")
if grpc_host is None:
    raise ValueError("GRPC_HOST environment variable is not set")

client = weaviate.connect_to_custom(
    http_host=http_host,
    http_port=8080,  # Default is 80, WCD uses 443
    http_secure=False,
    grpc_host=grpc_host,
    grpc_port=50051,  # Default is 50051, WCD uses 443
    grpc_secure=False,
    auth_credentials=Auth.api_key(
        weaviate_api_key
    ),  # The API key to use for authentication
    headers={"X-OpenAI-Api-Key": openai_api_key},
    skip_init_checks=True,
)

print("Client is live:", client.is_live())
print(client.collections.list_all().keys())  # List all collections in database
# print(client.collections.get("collection_name")) # View the configuration of a collection
# client.collections.delete("collection_name")  # THIS WILL DELETE THE SPECIFIED COLLECTION AND ITS OBJECTS
client.close()

## Scrape PDFs
To test locally, put PDFs in a directory named "pdfs" at the same level as this notebook. This part should output a list of Langchain Document objects, one for each PDF you put in the directory.  

In [ ]:
def load_and_scrape(pdf_directory: str) -> list[Document]:
    """Load and scrape a PDF into a langchain document object."""
    pdf_files = [f for f in os.listdir(pdf_directory) if f.endswith(".pdf")]

    if not pdf_files:
        print("No PDFs found in the directory.")
        return None

    documents = []

    for pdf_file in pdf_files:
        pdf_path = Path(pdf_directory) / pdf_file
        print(f"Loading PDF: {pdf_file}")

        loader = PyMuPDFLoader(pdf_path)
        document = loader.load()
        documents += document

    return documents

In [ ]:
# Test the scraper
# Note: this will print every document, so maybe test on a smaller doc first
pdf_directory = "./pdfs/"
documents = load_and_scrape(pdf_directory=pdf_directory)
for doc in documents:
    print(doc.page_content)

## Chunk documents
This next step will chunk the documents. It will also output a list of langchain Documents, but this list will be longer.

In [ ]:
def chunk_docs(
    docs: list[Document],
    chunk_size: int = 1000,
    chunk_overlap: int = 50,
) -> list[Document]:
    """Chunk langchain documents.

    Parameters
    ----------
        docs : list
            name of the list of langchain documents
        chunk_size : int
            size of chunks (in characters)
        chunk_overlap : int
            overlap of chunks (in characters)

    Returns
    -------
        docs : list
            list of langchain documents
    """
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, chunk_overlap=chunk_overlap
    )
    return text_splitter.split_documents(docs)

In [ ]:
# Test document chunker by printing out the first two chunks
chunked_docs = chunk_docs(
    docs=documents,
    chunk_size=1000,
    chunk_overlap=50,
)
for n in range(2):
    print(chunked_docs[n])

## Clean the Metadata

In [ ]:
def sanitize_metadata(docs : list[Document]) -> list[Document]:
    start_time = time.time()
    documents = []
    for doc in docs:
        # Check if page_content is None or not a string, and provide a default
        content = doc.page_content if doc.page_content is not None else ""

        # Clean metadata
        forbidden_keys = {'id', 'vector'}
        if hasattr(doc, 'metadata') and isinstance(doc.metadata, dict):
            cleaned_metadata = {k: v for k, v in doc.metadata.items() if k not in forbidden_keys}
        else:
            cleaned_metadata = {}

        # Create document with valid string content
        documents.append(Document(page_content=content, metadata=cleaned_metadata))
    end_time = time.time()
    duration = end_time - start_time
    print(f"Cleaned docs in {duration:.2f} seconds")
    return documents

In [ ]:
# Test document sanitizer by printing out the first two chunks
cleaned_docs = sanitize_metadata(chunked_docs)
for n in range(2):
    print(cleaned_docs[n])

## Upload to Weaviate
The last step of ingestion is to upload the documents to Weaviate. For this last step, you must reconnect to Weaviate. It is best to put that connection in a `try:` block, along with your other code. This way you can ensure the client will close even if an Error occurs. The first function is fine for small documents (anything that takes less than a minute to scrape), but you will want to use the batched version for any sizeable ingest.

In [ ]:
def push_docs_to_weaviate(
    docs: list[Document], index_name: str
) -> None:
    """Upload documents to Weaviate using the OpenAI embeddings."""
    embeddings = OpenAIEmbeddings(
        openai_api_key=openai_api_key,
        model="text-embedding-3-large",  # Default model text-embedding-ada-002
    )
    WeaviateVectorStore.from_documents(
        documents=docs,
        embedding=embeddings,
        index_name=index_name,
        client=client,
        text_key="page_content",
        attributes=list(docs[0].metadata.keys()),
    )

In [ ]:
def push_docs_to_weaviate_batched(
    docs: list[Document],
    index_name: str,
    batch_size: int = 200
) -> None:
    print("Pushing docs to Weaviate in batches")
    start_time = time.time()

    embeddings = OpenAIEmbeddings(
        openai_api_key=openai_api_key,
        model="text-embedding-ada-002"
    )

    for i in range(0, len(docs), batch_size):
        batch = docs[i:i + batch_size]
        print(f"Batch {i // batch_size + 1}: {len(batch)} docs")

        WeaviateVectorStore.from_documents(
            documents=batch,
            embedding=embeddings,
            index_name=index_name,
            client=client,
            text_key="page_content",
            attributes=list(batch[0].metadata.keys()) if batch[0].metadata else [],
        )

    end_time = time.time()
    print(f"Pushed all docs to Weaviate in {end_time - start_time:.2f} seconds")

### Test by Creating a Sample Database

In [ ]:
try:
    client = weaviate.connect_to_custom(
        http_host=http_host,
        http_port=8080,
        http_secure=False,
        grpc_host=grpc_host,
        grpc_port=50051,
        grpc_secure=False,
        auth_credentials=Auth.api_key(weaviate_api_key),
        headers={"X-OpenAI-Api-Key": openai_api_key},
        skip_init_checks=True,
    )
    push_docs_to_weaviate(cleaned_docs, "Full_PDF_Ingestion")
    print(client.collections.list_all().keys())
except Exception as e:
    print("Error:", e)
finally:
    client.close()

### Test that Database was Populated
Warning: This will print out your entire database, so start with a small example

In [ ]:
try:
    client = weaviate.connect_to_custom(
        http_host=http_host,
        http_port=8080,
        http_secure=False,
        grpc_host=grpc_host,
        grpc_port=50051,
        grpc_secure=False,
        auth_credentials=Auth.api_key(weaviate_api_key),
        headers={"X-OpenAI-Api-Key": openai_api_key},
        skip_init_checks=True,
    )
    collection_name = "Full_PDF_Ingestion"
    if collection_name not in client.collections.list_all():
        print(
            f"Collection '{collection_name}' does not exist. Run ingestion first."
        )

    collection = client.collections.get(collection_name)
    for item in collection.iterator():
        print(item.uuid, item.properties)
except Exception as e:
    print("Error:", e)
finally:
    client.close()

## Query the Database
Now, we can query the Weaviate database to test whether the RAG is working. Use the collection name you specified above (for this example I use 'Full_PDF_Ingestion'), and make sure it is printed in the "Test by Creating a Sample Database was Created" section, so that you know it has been ingested.

### Keyword Search
This first test will just search for a given keyword in your database.

In [ ]:
def bm25_search(keyword: str, collection_name: str) -> None:
    """Keyword search query."""
    collection = client.collections.get(collection_name)
    response = collection.query.bm25(query=keyword, limit=4)

    for o in response.objects:
        print(o.properties)  # Object properties

In [ ]:
try:
    client = weaviate.connect_to_custom(
        http_host=http_host,
        http_port=8080,
        http_secure=False,
        grpc_host=grpc_host,
        grpc_port=50051,
        grpc_secure=False,
        auth_credentials=Auth.api_key(weaviate_api_key),
        headers={"X-OpenAI-Api-Key": openai_api_key},
        skip_init_checks=True,
    )

    bm25_search(keyword="optimal", collection_name="Full_PDF_Ingestion")
except Exception as e:
    print("Error:", e)
finally:
    client.close()

### RAG Query
Now we can do the final test: a RAG based query using the generative AI query function provided by weaviate.

In [ ]:
def query_rag(question: str, collection_name: str) -> None:
    """Query Weaviate using the near vector search."""
    if collection_name not in client.collections.list_all():
        print(
            f"Collection '{collection_name}' does not exist. Run ingestion first."
        )
        return

    embedding_model = OpenAIEmbeddings(
        openai_api_key=openai_api_key, model="text-embedding-3-large"
    )
    query_vector = embedding_model.embed_query(question)

    collection = client.collections.get(collection_name)
    # print(collection.config.get())

    response = collection.generate.near_vector(
        near_vector=query_vector,
        limit=4,
        grouped_task=question,
        return_metadata=MetadataQuery(distance=True),
    )
    # print("RAW SEARCH RESPONSE:", response) # See what chunks are pulled
    print("RAG GENERATED RESPONSE:", response.generated)

In [ ]:
try:
    client = weaviate.connect_to_custom(
        http_host=http_host,
        http_port=8080,
        http_secure=False,
        grpc_host=grpc_host,
        grpc_port=50051,
        grpc_secure=False,
        auth_credentials=Auth.api_key(weaviate_api_key),
        headers={"X-OpenAI-Api-Key": openai_api_key},
        skip_init_checks=True,
    )

    query_rag(
        question="How many strongly lensed Type Ia supernovae is LSST expected to discover?",
        collection_name="Full_PDF_Ingestion",
    )
except Exception as e:
    print("Error:", e)
finally:
    client.close()